# Aula 05 — Agente ReAct: o loop pensar → agir → observar

**Objetivo:** entender a única novidade desta aula — UMA aresta (`tools ->
`assistant`) — e por que ela transforma o roteador da Aula 04 num agente de
verdade, capaz de N passos encadeados.

**Pré-requisitos:** Aula 03 (`tool_calls`) e Aula 04 (porteiro + caixa).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 1. Analogia: o atendente que volta ao caixa quantas vezes precisar

Na Aula 04 o cliente pedia **uma** conta e ia embora com o recibo cru. Agora
o fluxo é o do balcão real:

1. Cliente: 'some 3 e 4, depois multiplique por 2, e divida por 5'
2. Atendente anota, manda o **caixa** calcular `3+4` → recibo `7`
3. Atendente **lê o recibo** e manda o caixa calcular o próximo passo
4. Repete até não haver mais conta → responde a frase final

**Pensar** (ler histórico e decidir) → **agir** (executar tool) → **observar**
(ler o recibo). Esse ciclo se chama **ReAct** (*Reason + Act*). A diferença
para a Aula 04 é literalmente uma aresta a mais no desenho:

In [ ]:
# CÉLULA 2 — o mesmo grafo da Aula 04, MAIS uma aresta (100% offline)
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

def add(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

# Fake com 2 falas: 1º pede a tool, depois encerra com frase
fake = FakeMessagesListChatModel(responses=[
    AIMessage(content='', tool_calls=[
        {'name': 'add', 'args': {'a': 3, 'b': 4}, 'id': 'c1', 'type': 'tool_call'}]),
    AIMessage(content='O resultado é 7.', tool_calls=[]),
])

def assistant(state: MessagesState) -> dict:
    return {'messages': [fake.invoke(state['messages'])]}

b = StateGraph(MessagesState)
b.add_node('assistant', assistant)
b.add_node('tools', ToolNode([add]))
b.add_edge(START, 'assistant')
b.add_conditional_edges('assistant', tools_condition)
b.add_edge('tools', 'assistant')   # <-- A NOVIDADE: a volta que cria o loop
reagente = b.compile()

out = reagente.invoke({'messages': [HumanMessage(content='Some 3 e 4')]})
for m in out['messages']:
    print('[' + type(m).__name__ + ']', repr(m.content))
assert [type(m).__name__ for m in out['messages']] == ['HumanMessage', 'AIMessage', 'ToolMessage', 'AIMessage']
print('OK — 4 mensagens: pergunta, pedido, recibo e resposta final.')

## 2. Conceito: o ciclo e quando ele para

```
START -> assistant --tools_condition--> tools
              ^                           |
              +------ (a volta!) ----------+
              |
              +--- (sem tool_calls) ---> END
```

- Cada volta completa (`assistant → tools → assistant`) é **1 passo ReAct**:
  o LLM vê tudo que aconteceu (o histórico acumula via `add_messages`) e
  decide o próximo movimento.
- **Condição de parada:** o LLM responde **sem** `tool_calls` → o porteiro
  devolve `'__end__'` → acabou. Não há contador de passos no grafo — quem
  decide parar é o próprio LLM.
- Por isso existe a **trava de segurança** `recursion_limit` (padrão: 25): se
  o LLM nunca parar de pedir tools, o LangGraph aborta com `GraphRecursionError`
  em vez de rodar para sempre (seção 5).
- **`SystemMessage`**: instrução fixa ('você faz contas, responda em
  português'). Nesta aula ela vai **só na chamada** (`[SYS] + messages`), sem
  ser gravada no estado — assim não polui o histórico a cada volta.

In [ ]:
# CÉLULA 3 — o agente completo com Groq real (Fake: repete a demo)
from langchain_core.messages import SystemMessage

SISTEMA = 'Você é um assistente que faz contas. Responda em português.'
MODELO = 'openai/gpt-oss-20b'

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

def divide(a: float, b: float) -> float:
    """Divide a por b."""
    return a / b

TOOLS = [add, multiply, divide]

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools(TOOLS)
    print('AVISO: modo Fake offline (repete a demo da CÉLULA 2).')
    return fake

def assistant_llm(state: MessagesState) -> dict:
    return {'messages': [get_llm().invoke(
        [SystemMessage(content=SISTEMA)] + state['messages'])]}

ag = StateGraph(MessagesState)
ag.add_node('assistant', assistant_llm)
ag.add_node('tools', ToolNode(TOOLS))
ag.add_edge(START, 'assistant')
ag.add_conditional_edges('assistant', tools_condition)
ag.add_edge('tools', 'assistant')
agente = ag.compile()

if os.getenv('GROQ_API_KEY'):
    saida = agente.invoke({'messages': [HumanMessage(
        content='Some 3 e 4, depois multiplique por 2, e finalmente divida por 5. Mostre o resultado.')]})
    for m in saida['messages']:
        print('[' + type(m).__name__ + ']', repr(m.content), getattr(m, 'tool_calls', ''))
    final = saida['messages'][-1].content
    assert '2,8' in final or '2.8' in final, final
    print('OK — 3 contas encadeadas, resposta final em frase.')
else:
    print('(sem chave: a CÉLULA 2 já mostrou o ciclo no Fake;')
    print(' com a chave Groq, esta célula encadeia 3 contas de verdade.)')

## 3. Dissecando a volta (linha a linha do que mudou)

- `b.add_edge('tools', 'assistant')` — **a única linha nova** em relação à
  Aula 04 (que tinha `tools -> END`). Tudo o mais é idêntico: mesmos nós,
  mesma condicional, mesmo estado.
- `assistant` relê o estado **inteiro** a cada volta — pergunta original +
  pedidos + recibos. É assim que o LLM 'lembra' do `7` ao pedir a próxima conta.
- `[SystemMessage(content=SISTEMA)] + state['messages']` — o SYS entra só na
  chamada. Alternativa válida: gravar o SYS no estado no 1º turno (ver seção 5c).
- O `FakeMessagesListChatModel` com 2 falas simula as 2 passagens pelo `assistant`:
  1ª volta pede a tool, 2ª volta encerra. Com Groq real, quantas voltas houver
  quem decide é o LLM — por isso o traço real (CÉLULA 4) pode surpreender.

In [ ]:
# CÉLULA 4 — lendo o traço real: o LLM decide a decomposição
# Rode com a chave Groq e compare com o que VOCE faria em 3 passos.
if os.getenv('GROQ_API_KEY'):
    n_tools = sum(1 for m in saida['messages'] if type(m).__name__ == 'ToolMessage')
    print('ToolMessages executados:', n_tools)
    for m in saida['messages']:
        if getattr(m, 'tool_calls', None):
            print('pediu:', m.tool_calls[0]['name'], m.tool_calls[0]['args'])
        if type(m).__name__ == 'ToolMessage':
            print('recibo:', repr(m.content))
    print()
    print('Repare: o modelo pode ter pulado o x2 (fez 7x2=14 de cabeça) e ido')
    print('direto ao divide(14, 5). A decomposição é dele — o resultado é que importa.')
else:
    print('(sem chave: o traço do Fake tem 1 pedido + 1 recibo, CÉLULA 2.)')

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Sem a volta, o recibo morre sem resposta (Aula 04 de novo)

O mesmo Fake, o mesmo pedido — só muda a aresta final para `END`. O LLM nunca
relê o `'7'`: o traço termina no recibo cru.

In [ ]:
# CÉLULA 5 — contraste: tools -> END (100% offline)
b_sem = StateGraph(MessagesState)
b_sem.add_node('assistant', assistant)
b_sem.add_node('tools', ToolNode([add]))
b_sem.add_edge(START, 'assistant')
b_sem.add_conditional_edges('assistant', tools_condition)
b_sem.add_edge('tools', END)   # <- a única diferença para a CÉLULA 2
out_sem = b_sem.compile().invoke({'messages': [HumanMessage(content='Some 3 e 4')]})
print([type(m).__name__ for m in out_sem['messages']])
assert [type(m).__name__ for m in out_sem['messages']] == ['HumanMessage', 'AIMessage', 'ToolMessage']
print('OK — 3 mensagens, sem frase final. Uma aresta decide entre agente e roteador.')

### 4b. Trava de segurança: `recursion_limit` e o loop sem fim

Se o LLM nunca responder sem `tool_calls` (bug de prompt, tool que sempre
provoca novo pedido), o grafo rodaria para sempre. O LangGraph conta os passos
(padrão: 25) e aborta com `GraphRecursionError`. Abaixo, um Fake 'teimoso' que
sempre pede tool, com limite 3 — 100% offline e determinístico.

In [ ]:
# CÉLULA 6 — loop infinito abortado (100% offline)
from langgraph.errors import GraphRecursionError

teimoso = FakeMessagesListChatModel(responses=[AIMessage(content='', tool_calls=[
    {'name': 'add', 'args': {'a': 1, 'b': 1}, 'id': 'c9', 'type': 'tool_call'}])])

def assistant_teimoso(state: MessagesState) -> dict:
    return {'messages': [teimoso.invoke(state['messages'])]}

bt = StateGraph(MessagesState)
bt.add_node('assistant', assistant_teimoso)
bt.add_node('tools', ToolNode([add]))
bt.add_edge(START, 'assistant')
bt.add_conditional_edges('assistant', tools_condition)
bt.add_edge('tools', 'assistant')
try:
    bt.compile().invoke({'messages': [HumanMessage(content='Some 1 e 1')]},
                        config={'recursion_limit': 3})
    print('terminou?! (não deveria)')
except GraphRecursionError as erro:
    print('GraphRecursionError:', str(erro)[:90])
    print('OK — todo agente precisa de limite; o padrão (25) vale para produção.')

### 4c. `SystemMessage` gravado no estado (alternativa válida)

Em vez de prependar o SYS a cada chamada, grave-o como 1ª mensagem do estado:
ele viaja no histórico como as outras. Diferença prática: gravado, ele ocupa
1 posição fixa e aparece nos logs; prependado, é invisível no estado. Para
conversas longas, prependar evita que o SYS seja podado junto (Aula 08).

In [ ]:
# CÉLULA 7 — SYS no estado (Groq real; Fake: repete a lógica)
if os.getenv('GROQ_API_KEY'):
    out_sys = agente.invoke({'messages': [
        SystemMessage(content=SISTEMA),
        HumanMessage(content='Quanto é 2+2?')]})
    print('final:', repr(out_sys['messages'][-1].content))
    print('tipos:', [type(m).__name__ for m in out_sys['messages']])
else:
    print('(sem chave: SYS no estado funciona igual — o primeiro item do')
    print(' histórico passa a ser SystemMessage em vez de HumanMessage.)')

## 5. Erros clássicos

1. **Esquecer a volta.** Sem `tools -> assistant`, você tem a Aula 04: recibo
   cru, sem frase final. Uma aresta decide entre roteador e agente.
2. **Achar que o agente 'conta passos'.** Quem decide parar é o LLM (resposta
   sem `tool_calls`). Sem trava, loop infinito é possível — daí o
   `recursion_limit` (CÉLULA 6).
3. **Esperar decomposição fixa.** O modelo pode pular passos mentais (fez
   `7x2=14` de cabeça). Afirme o **resultado final**, não a sequência de tools.
4. **SYS duplicado no estado.** Se você grava o SYS no estado E prepende na
   chamada, ele aparece 2x para o LLM. Escolha um dos dois.
5. **Histórico sem `add_messages`.** Sem reducer, o recibo apagaria a pergunta
   (Aula 03). ReAct sem memória de passos não funciona.
6. **Tool lenta/bloqueante sem timeout.** Cada volta espera a tool. Tool de rede
   sem timeout trava o agente inteiro — trate como código de produção.

## 6. Exercício guiado — a conta da lanchonete

Situação: 'Comprei 2 pastéis de R$ 8 e 1 refri de R$ 6. Qual o total?'
Resposta esperada: `2x8+6 = 22`. O agente deve encadear `multiply` e `soma`
sozinho e terminar com frase contendo `22`.

In [ ]:
# CÉLULA 8 — exercício guiado (conta da lanchonete)
if os.getenv('GROQ_API_KEY'):
    pedido = 'Comprei 2 pasteis de R$ 8 e 1 refri de R$ 6. Qual o total? Responda com o valor.'
    out_lanche = agente.invoke({'messages': [HumanMessage(content=pedido)]})
    for m in out_lanche['messages']:
        print('[' + type(m).__name__ + ']', repr(m.content), getattr(m, 'tool_calls', ''))
    assert '22' in out_lanche['messages'][-1].content, out_lanche['messages'][-1].content
    print('OK — multiply + soma encadeados, total 22 na frase final.')
else:
    print('(sem chave: o Fake desta aula só conhece some-3-e-4;')
    print(' com a chave Groq, o agente encadeia multiply(2,8) + soma(16,6) = 22.)')

## 7. Desafio — 3 contas, 3 tools, 1 frase

'Tenho R$ 100. Comprei 3 pastéis de R$ 8, 2 refris de R$ 6 e dividi a conta
com um amigo. Quanto cada um paga?' (Esperado: `(24+12)/2 = 18`.)

- Com a chave Groq: rode no `agente` e confira `'18'` na frase final, com
  pelo menos 2 `ToolMessage` no traço.
- **Pegadinha (pense antes):** e se o modelo fizer `36/2` de cabeça e pedir só
  2 tools? O resultado continua certo — afirme o final, não o caminho.
- Sem chave: desenhe no papel o traço ideal (pergunta → pedido → recibo →
  pedido → recibo → frase) e compare com a CÉLULA 2.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Qual é a única aresta que separa o roteador (Aula 04) do agente ReAct?
- [ ] O que é 1 passo ReAct e quem decide quando parar?
- [ ] Para que serve o `recursion_limit` e qual erro ele gera?
- [ ] Onde o `SystemMessage` vai: no estado, na chamada, ou tanto faz? Quais os trade-offs?
- [ ] Por que afirmar o resultado final e não a sequência de tools?
- [ ] No seu exemplo do pastel/refri/conta, quais voltas o agente daria?

**Próxima aula (06 — Memória):** cada `invoke` hoje esquece tudo. `MemorySaver`
+ `thread_id` transformam chamadas isoladas em conversa — 'isso=7' lembrado no
turno seguinte.